# Newberry 3-D superhot demonstration: 400 °C

This notebook uses only geoPFA's public, configuration-driven workflow. The configuration is visible in the notebook; local source data and generated outputs are deliberately not version-controlled. The probabilistic target map is compared with the traditional `VoterVeto` workflow.

The heat term is the thermal-model exceedance probability `P(T > 400 °C)` and is deliberately **not** updated with available 150 °C proxy labels. Reservoir and insulation evidence coefficients are propagated from explicit priors because target-matched labels do not support posterior updating.

In [ ]:
import copy
import json
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from geopfa.layer_combination import VoterVeto
from geopfa.prob import ProbabilisticConfig, run_probabilistic

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise FileNotFoundError("geoPFA repository root not found")


def required_local_path(env_name: str, default: Path) -> Path:
    path = Path(os.environ.get(env_name, default)).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(
            f"Required local input is absent: {path}. Set {env_name} or prepare "
            "the documented public study data before running this notebook."
        )
    return path

repo_root = find_repo_root(Path.cwd())
project_dir = repo_root / "examples" / "Newberry" / "3D"
output_root = Path(
    os.environ.get("GEOPFA_DEMO_OUTPUT_ROOT", project_dir / "outputs")
).expanduser().resolve()
output_dir = output_root / "newberry_superhot_400c"
pfa_path = required_local_path(
    "GEOPFA_NEWBERRY_PFA", project_dir / "notebooks" / "fpa.pkl"
)
with pfa_path.open("rb") as stream:
    pfa = pickle.load(stream)

config_dict = {'enabled': True,
 'output_dir': '../outputs/superhot_400c',
 'dimensions': '3d',
 'labels': {'observation_models': {'heat': {'family': 'gaussian'}}},
 'alpha': {'heat': {'mode': 'thermal_layer_exceedance',
                    'layer': 'temperature_model_500m',
                    'threshold': 400.0,
                    'uncertainty_column': 'temperature_predictive_sd_c',
                    'p_min': 1e-12,
                    'p_max': 0.999999999999,
                    'force_prior_predictive': True,
                    'use_evidence_prior': False},
           'reservoir': {'mode': 'scalar',
                         'scalar_fallback_pr0': 0.5,
                         'force_prior_predictive': True,
                         'use_evidence_prior': True},
           'insulation': {'mode': 'scalar',
                          'scalar_fallback_pr0': 0.5,
                          'force_prior_predictive': True,
                          'use_evidence_prior': True}},
 'evidence': {'standardization': 'prediction_support',
              'include_layers': ['density_joint_inv',
                                 'mt_resistivity_joint_inv',
                                 'temperature_model_500m',
                                 'earthquakes',
                                 'ring_faults',
                                 'lineaments'],
              'regularization': {'prior_means': {'reservoir:density_joint_inv': 0.6931471805599453,
                                                 'reservoir:mt_resistivity_joint_inv': 0.6931471805599453,
                                                 'reservoir:earthquakes': 0.6931471805599453,
                                                 'reservoir:ring_faults': 0.6931471805599453,
                                                 'reservoir:lineaments': 0.6931471805599453,
                                                 'insulation:density_joint_inv': 0.6931471805599453,
                                                 'insulation:mt_resistivity_joint_inv': 0.6931471805599453,
                                                 'insulation:earthquakes': 0.6931471805599453,
                                                 'insulation:temperature_model_500m': 0.6931471805599453},
                                 'prior_precisions': {'reservoir:density_joint_inv': 4.0,
                                                      'reservoir:mt_resistivity_joint_inv': 4.0,
                                                      'reservoir:earthquakes': 4.0,
                                                      'reservoir:ring_faults': 4.0,
                                                      'reservoir:lineaments': 4.0,
                                                      'insulation:density_joint_inv': 4.0,
                                                      'insulation:mt_resistivity_joint_inv': 4.0,
                                                      'insulation:earthquakes': 4.0,
                                                      'insulation:temperature_model_500m': 4.0}}},
 'spatial_field': {'enabled': False},
 'inference': {'backend': 'gblk',
               'gblk_bayesian': {'enabled': True,
                                 'n_draws': 512,
                                 'seed': 20260909,
                                 'ci_level': 0.95,
                                 'cluster_effect': False}},
 'calibration': {'method': 'none'},
 'combination': {'rule': 'product'},
 'scenarios': [],
 'outputs': {'posterior_draw_blocks': True,
             'posterior_draw_block_size': 32,
             'format': ['parquet']}}
config_dict["output_dir"] = str(output_dir)
config = ProbabilisticConfig.from_dict(config_dict)

# The traditional workflow uses the repository's standard processed config.
vv_config_path = project_dir / "config" / "newberry_superhot_processed_config.json"
vv_config = json.loads(vv_config_path.read_text(encoding="utf-8"))
pfa_vv = copy.deepcopy(pfa)
pfa_vv["criteria"]["geologic"]["weight"] = vv_config["criteria"]["geologic"]["weight"]
for component_name, component in pfa_vv["criteria"]["geologic"]["components"].items():
    component_config = vv_config["criteria"]["geologic"]["components"][component_name]
    component["weight"] = component_config["weight"]
    component["pr0"] = component_config["pr0"]
    for layer_name, layer in component["layers"].items():
        layer["weight"] = component_config["layers"][layer_name]["weight"]
config

In [ ]:
pfa_vv = VoterVeto.do_voter_veto(
    pfa_vv,
    normalize_method="minmax",
    component_veto=False,
    criteria_veto=True,
    normalize=True,
    norm_to=5,
)
vv_volume = pfa_vv["criteria"]["geologic"]["components"]["heat"]["pr_norm"]
model_result = run_probabilistic(
    pfa,
    config,
    input_artifacts={"pfa_pickle": pfa_path},
)
heat_probability = model_result.components["heat"].probability
combined_probability = model_result.combined
if len(heat_probability) != 439_198:
    raise ValueError(
        f"expected 439,198 supported Newberry cells, found {len(heat_probability):,}"
    )
heat_probability[["probability", "probability_lo", "probability_hi"]].describe()

In [ ]:
z = heat_probability.geometry.z.to_numpy()
depth_table = heat_probability.assign(z=z).groupby("z", sort=True)["probability"].max()
display_z = float(depth_table.idxmax())
mask = np.isclose(z, display_z)
heat_slice = heat_probability.loc[mask]
combined_slice = combined_probability.loc[mask]
combined_vmax = float(combined_probability.probability.quantile(0.99))
vv_heat = vv_volume.loc[np.isclose(vv_volume.geometry.z.to_numpy(), display_z)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
vv_heat.plot(column="favorability", cmap="viridis", vmin=0, vmax=5, markersize=3, legend=True, ax=axes[0])
heat_slice.plot(column="probability", cmap="magma", vmin=0, vmax=1, markersize=3, legend=True, ax=axes[1])
combined_slice.plot(column="probability", cmap="magma", vmin=0, vmax=combined_vmax, markersize=3, legend=True, ax=axes[2])
axes[0].set_title("Traditional VoterVeto heat score")
axes[1].set_title("Thermal probability: P(T > 400 °C)")
axes[2].set_title("Combined target (color clipped at global 99th percentile)")
for axis in axes:
    axis.set_axis_off()
fig.suptitle(f"Newberry 3-D slice at elevation {display_z:,.0f} m MSL")
plt.show()

In [ ]:
{
    "target": "P(T > 400 C) in the Newberry 3-D volume",
    "n_grid_cells": int(len(heat_probability)),
    "display_elevation_m_msl": display_z,
    "heat_probability": heat_probability.probability.describe().to_dict(),
    "combined_probability": combined_probability.probability.describe().to_dict(),
    "assessment_scope": (
        "The public thermal mean field is paired with a provenance-bound predictive SD "
        "derived from held-out residual assessment. That assessment supports the "
        "uncertainty construction but is not target-matched 400 C calibration. No "
        "population calibration claim is made for 400 C because no target-matched "
        "labels exist."
    ),
}

## Interpretation and caveat

This is a low-data Bayesian prior-predictive result: geoPFA propagates a physically interpretable thermal probability and explicit parameter-prior uncertainty through a real 3-D target volume without pretending that 150 °C proxy labels validate a 400 °C endpoint. The map is a screening product. It is not proof of a drillable reservoir and has no target-matched population calibration estimate.